## Imports and Global Variables

In [3]:
import re
import os
import cv2
import pickle
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import random
import tensorflow as tf

# --- 1. CONFIGURATION ---
# GPU is implicitly enabled by NOT setting CUDA_VISIBLE_DEVICES to -1
DATASET_ROOT = "/kaggle/input/vimeo-90k-1/vimeo_settuplet_1/sequences"
PICKLE_PATH = "septuplets.pkl"
OUTPUT_DIR = "/kaggle/working/checkpoints"
VISUAL_DIR = "/kaggle/working/visual_tests"
CROP_SIZE = (256, 256)

# Create output directories
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
if not os.path.exists(VISUAL_DIR): os.makedirs(VISUAL_DIR)

print("hi")

hi


## Creating Persistence with Pickle

In [4]:
# --- STEP 3: DATA INDEXING (PHASE 1) ---
def generate_septuplet_pickle(dataset_path, output_pkl):
    """
    Walks through the vimeo-90k-1 directory structure once to index all septuplets.
    Structure: sequences / {video_id} / {folder_id} / im1.png...im7.png
    """
    if os.path.exists(output_pkl):
        print(f"✅ Index found at {output_pkl}. Loading...")
        with open(output_pkl, 'rb') as f:
            data = pickle.load(f)
            print(f"Loaded {len(data)} septuplet sequences.")
            return data
    
    print("🔍 No index found. Walking directory (this may take a few minutes)...")
    all_septuplets = []
    
    if not os.path.exists(dataset_path):
        print(f"❌ Error: Dataset path {dataset_path} not found!")
        return []

    # Iterate through video folders (e.g., 00001, 00002...)
    video_folders = sorted(os.listdir(dataset_path))
    for video_id in video_folders:
        video_path = os.path.join(dataset_path, video_id)
        if not os.path.isdir(video_path): continue
        
        # Iterate through sequence folders (e.g., 0001, 0002...)
        seq_folders = sorted(os.listdir(video_path))
        for seq_id in seq_folders:
            seq_path = os.path.join(video_path, seq_id)
            
            # Construct the list of 7 image paths for this septuplet
            septuplet_paths = [os.path.join(seq_path, f"im{i}.png") for i in range(1, 8)]
            
            # Verify integrity: ensure the sequence folder is complete
            # We check the first, middle, and last to be fast
            if os.path.exists(septuplet_paths[0]) and \
               os.path.exists(septuplet_paths[3]) and \
               os.path.exists(septuplet_paths[6]):
                all_septuplets.append(septuplet_paths)
                
    print(f"✨ Found {len(all_septuplets)} septuplet sequences.")
    
    # Save the list to the pickle file for future sessions
    with open(output_pkl, 'wb') as f:
        pickle.dump(all_septuplets, f)
    print(f"💾 septuplets.pkl saved to current directory.")
    
    return all_septuplets

# Execute indexing logic
SEPTUPLET_LIST = generate_septuplet_pickle(DATASET_ROOT, PICKLE_PATH)

print("\n--- SETUP COMPLETE ---")
print(f"Total sequences indexed: {len(SEPTUPLET_LIST)}")
if len(SEPTUPLET_LIST) > 0:
    print(f"Sample path: {SEPTUPLET_LIST[0][0]}")

✅ Index found at septuplets.pkl. Loading...
Loaded 9953 septuplet sequences.

--- SETUP COMPLETE ---
Total sequences indexed: 9953
Sample path: /kaggle/input/vimeo-90k-1/vimeo_settuplet_1/sequences/00001/0001/im1.png


## Image Processing

In [5]:
def load_and_crop_septuplet(paths, crop_size=CROP_SIZE):
    """
    Loads 7 images and applies the SAME random crop to all of them.
    Returns: A numpy array of shape (7, H, W, 3) normalized to [0, 1]
    """
    imgs = []
    # 1. Load all images first
    for p in paths:
        img = cv2.imread(p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        imgs.append(img)
    
    # 2. Determine random crop coordinates
    h, w, _ = imgs[0].shape
    ch, cw = crop_size
    
    x = random.randint(0, w - cw)
    y = random.randint(0, h - ch)
    
    # 3. Apply crop to all
    cropped_imgs = [img[y:y+ch, x:x+cw] for img in imgs]
    
    # 4. Convert to float32 and normalize
    return np.array(cropped_imgs).astype('float32') / 255.0

In [6]:
class SeptupletGenerator(keras.utils.Sequence):
    """
    Keras Sequence generator for Multi-Head training.
    """
    def __init__(self, septuplet_list, batch_size=BATCH_SIZE, shuffle=True):
        self.septuplet_list = septuplet_list
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.septuplet_list) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            random.shuffle(self.septuplet_list)

    def __getitem__(self, index):
        batch_paths = self.septuplet_list[index * self.batch_size : (index + 1) * self.batch_size]
        
        input_data = []      # Shape: (Batch, 256, 256, 18)
        target_pred = []     # Target for Head 1 (im7)
        target_interp = []   # Target for Head 2 (im4)

        for paths in batch_paths:
            # Load the 7 frames with consistent cropping
            seq = load_and_crop_septuplet(paths)
            
            # --- TASK 1: PREDICTION HEAD (1,2,3,4,5,6 -> 7) ---
            # Input is im1-im6 concatenated on channel axis
            pred_input = np.concatenate(seq[0:6], axis=-1)
            
            # --- TASK 2: INTERPOLATION HEAD (1,2,3,5,6,7 -> 4) ---
            # Input is im1,2,3 AND im5,6,7 concatenated
            interp_input = np.concatenate([seq[0], seq[1], seq[2], seq[4], seq[5], seq[6]], axis=-1)
            
            # We use the SAME input tensor for the model (im1-6) but the generator
            # must provide the two distinct ground truths
            input_data.append(pred_input)
            target_pred.append(seq[6])   # im7
            target_interp.append(seq[3]) # im4

        # Keras expects: (Inputs, {"head_name1": target1, "head_name2": target2})
        return np.array(input_data), {
            "prediction_head": np.array(target_pred),
            "interpolation_head": np.array(target_interp)
        }

NameError: name 'BATCH_SIZE' is not defined

In [ ]:
# Initialize Generator
train_gen = SeptupletGenerator(SEPTUPLET_LIST, batch_size=BATCH_SIZE)

print("\n--- PHASE 2 COMPLETE ---")
print(f"Generator ready with {len(train_gen)} batches per epoch.")

## Clean U-Net Architecture

In [ ]:
def build_unet_block(x, filters, kernel_size=3, strides=1, downsample=False):
    """Standard U-Net building block: Conv -> BatchNormalization -> ReLU"""
    if downsample:
        x = layers.Conv2D(filters, kernel_size, strides=2, padding='same')(x)
    else:
        x = layers.Conv2D(filters, kernel_size, strides=strides, padding='same')(x)
    
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

In [ ]:
def build_multi_head_unet(input_shape=(None, None, 18)):
    """
    Constructs a clean, standard U-Net from scratch optimized for 
    Multi-Head Septuplet processing.
    """
    inputs = layers.Input(shape=input_shape, name="septuplet_input")

    # --- ENCODER (Downsampling) ---
    c1 = build_unet_block(inputs, 64)
    c1 = build_unet_block(c1, 64)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = build_unet_block(p1, 128)
    c2 = build_unet_block(c2, 128)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = build_unet_block(p2, 256)
    c3 = build_unet_block(c3, 256)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    # --- BRIDGE ---
    b1 = build_unet_block(p3, 512)
    b1 = build_unet_block(b1, 512)

    # --- DECODER (Upsampling) ---
    u1 = layers.Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(b1)
    u1 = layers.concatenate([u1, c3])
    u1 = build_unet_block(u1, 256)

    u2 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(u1)
    u2 = layers.concatenate([u2, c2])
    u2 = build_unet_block(u2, 128)

    u3 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(u2)
    u3 = layers.concatenate([u3, c1])
    u3 = build_unet_block(u3, 64)

    # --- DUAL HEADS ---
    # Prediction Head: Aiming for im7
    pred_head = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same', name='prediction_head')(u3)
    
    # Interpolation Head: Aiming for im4
    interp_head = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same', name='interpolation_head')(u3)

    model = keras.Model(inputs=inputs, outputs=[pred_head, interp_head])
    return model

In [ ]:
multi_head_vfi = build_multi_head_unet()
multi_head_vfi.summary()

print("\n--- PHASE 3 COMPLETE ---")
print("New Multi-Head U-Net constructed from scratch.")

## Custom Losses

In [ ]:
def ssim_loss(y_true, y_pred):
    """
    Calculates Structural Similarity loss.
    1.0 means perfect match, 0.0 means no match.
    We return (1 - SSIM) so that '0' is the goal for the optimizer.
    """
    # Max val is 1.0 because our generator normalizes images to [0, 1]
    return 1 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

In [ ]:
def combined_multi_task_loss(y_true, y_pred):
    """
    Calculates a weighted error for both heads.
    Standard L1 (MAE) is used for stability + SSIM for sharpness.
    """
    # 1. Prediction Head Loss (im7)
    pred_true = y_true['prediction_head']
    pred_out = y_pred['prediction_head']
    l1_pred = tf.reduce_mean(tf.abs(pred_true - pred_out))
    ssim_pred = ssim_loss(pred_true, pred_out)
    total_pred = (0.5 * l1_pred) + (0.5 * ssim_pred)

    # 2. Interpolation Head Loss (im4)
    interp_true = y_true['interpolation_head']
    interp_out = y_pred['interpolation_head']
    l1_interp = tf.reduce_mean(tf.abs(interp_true - interp_out))
    ssim_interp = ssim_loss(interp_true, interp_out)
    total_interp = (0.5 * l1_interp) + (0.5 * ssim_interp)

    # 3. Weighted Sum (60% Prediction / 40% Interpolation)
    return (0.6 * total_pred) + (0.4 * total_interp)

In [ ]:
import tensorflow as tf
from tensorflow import keras
import os

# --- SELF-CONTAINED LOSS FUNCTIONS ---

@keras.utils.register_keras_serializable(package="Custom")
def prediction_loss(y_true, y_pred):
    import tensorflow as tf # Local import prevents NameError during distributed load
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred))
    # We call the existing ssim_loss function
    ssim = 1 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
    return 0.5 * l1 + 0.5 * ssim

@keras.utils.register_keras_serializable(package="Custom")
def interpolation_loss(y_true, y_pred):
    import tensorflow as tf # Local import prevents NameError during distributed load
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred))
    ssim = 1 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
    return 0.5 * l1 + 0.5 * ssim

In [ ]:
print("\n--- PHASE 4 COMPLETE ---")
print("Custom SSIM and Multi-Task loss functions defined.")

## Checkpoints and visual testing

In [ ]:
def get_latest_checkpoint(checkpoint_dir):
    """Finds the checkpoint with the highest epoch number."""
    files = [f for f in os.listdir(checkpoint_dir) if f.endswith(".keras")]
    if not files: return None, 0
    
    epochs = [int(re.findall(r"epoch_(\d+)", f)[0]) for f in files if re.findall(r"epoch_(\d+)", f)]
    if not epochs: return None, 0
    
    latest_epoch = max(epochs)
    latest_file = os.path.join(checkpoint_dir, f"vfi_septuplet_epoch_{latest_epoch:02d}.keras")
    return latest_file, latest_epoch

In [ ]:
class VisualValidationCallback(keras.callbacks.Callback):
    """
    Every 2 epochs, picks a random septuplet, predicts frames, and saves images.
    """
    def __init__(self, generator, visual_dir=VISUAL_DIR):
        super().__init__()
        self.generator = generator
        self.visual_dir = visual_dir

    def on_epoch_end(self, epoch, logs=None):
        # Trigger every 2 epochs (e.g., epoch 1 (index 0) skips, epoch 2 (index 1) runs)
        if (epoch + 1) % 2 == 0:
            print(f"\n🎨 Running Visual Validation for Epoch {epoch+1}...")
            idx = random.randint(0, len(self.generator) - 1)
            x_batch, y_batch = self.generator[idx]
            
            # Predict
            predictions = self.model.predict(x_batch, verbose=0)
            
            # Extract first sample from batch
            # predictions[0] is prediction_head, predictions[1] is interpolation_head
            im7_pred = (predictions[0][0] * 255).astype(np.uint8)
            im4_pred = (predictions[1][0] * 255).astype(np.uint8)
            
            im7_gt = (y_batch['prediction_head'][0] * 255).astype(np.uint8)
            im4_gt = (y_batch['interpolation_head'][0] * 255).astype(np.uint8)
            
            # Save (Convert RGB to BGR for OpenCV)
            cv2.imwrite(os.path.join(self.visual_dir, f"e{epoch+1}_im7_pred.png"), cv2.cvtColor(im7_pred, cv2.COLOR_RGB2BGR))
            cv2.imwrite(os.path.join(self.visual_dir, f"e{epoch+1}_im7_gt.png"), cv2.cvtColor(im7_gt, cv2.COLOR_RGB2BGR))
            cv2.imwrite(os.path.join(self.visual_dir, f"e{epoch+1}_im4_pred.png"), cv2.cvtColor(im4_pred, cv2.COLOR_RGB2BGR))
            cv2.imwrite(os.path.join(self.visual_dir, f"e{epoch+1}_im4_gt.png"), cv2.cvtColor(im4_gt, cv2.COLOR_RGB2BGR))
            print("✅ Visuals Saved.")

In [ ]:
def get_callbacks(train_gen):
    """Returns a list of all callbacks needed for the run."""
    
    # 1. Standard Checkpointing (Saves after every epoch)
    checkpoint_path = os.path.join(OUTPUT_DIR, "vfi_septuplet_epoch_{epoch:02d}.keras")
    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        save_weights_only=False,
        verbose=1
    )
    
    # 2. Visual Validation (Custom)
    visual_cb = VisualValidationCallback(train_gen)
    
    # 3. Early Stopping (Optional but recommended to prevent wasted GPU time)
    early_stop = keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
    
    return [checkpoint_cb, visual_cb, early_stop]

In [ ]:
print("\n--- PHASE 5 COMPLETE ---")
print("Checkpointing and Visual Validation callbacks are ready.")
print("The pipeline is now fully designed and ready for GPU activation.")

## The Run

In [ ]:
print("🚀 INITIALIZING SINGLE-GPU (P100) TRAINING PIPELINE...")

# No MirroredStrategy here - Using default device (GPU:0)
# P100 has 16GB, so we can use a healthy batch size
BATCH_SIZE = 8 

# Initialize Data
SEPTUPLET_LIST = generate_septuplet_pickle(DATASET_ROOT, PICKLE_PATH)
train_gen = SeptupletGenerator(SEPTUPLET_LIST, batch_size=BATCH_SIZE)

# Check for Resume Point
latest_ckpt, start_epoch = get_latest_checkpoint(OUTPUT_DIR)

# Model setup (No strategy.scope needed)
if latest_ckpt:
    print(f"🔄 RESUMING from {latest_ckpt} at epoch {start_epoch}...")
    model = keras.models.load_model(
        latest_ckpt, 
        custom_objects={
            "ssim_loss": ssim_loss,
            "prediction_loss": prediction_loss,
            "interpolation_loss": interpolation_loss
        },
        safe_mode=False 
    )
    # Re-compile to apply the NaN-resistant loss and clean state
    model.compile(
        optimizer=model.optimizer,
        loss={
            "prediction_head": prediction_loss,
            "interpolation_head": interpolation_loss
        },
        loss_weights={"prediction_head": 0.6, "interpolation_head": 0.4}
    )
else:
    print("🆕 Starting NEW training session...")
    model = build_multi_head_unet()
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss={
            "prediction_head": prediction_loss,
            "interpolation_head": interpolation_loss
        },
        loss_weights={"prediction_head": 0.6, "interpolation_head": 0.4}
    )

# --- TRACKING THE TREND ---
# This file will store your loss history
log_path = os.path.join(OUTPUT_DIR, "training_log.csv")

checkpoint_path = os.path.join(OUTPUT_DIR, "vfi_septuplet_epoch_{epoch:02d}.keras")
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path, 
        verbose=1,
        save_weights_only=False
    ),
    keras.callbacks.CSVLogger(log_path, append=True), # Tracks loss reduction trend
    VisualValidationCallback(train_gen)
]

print(f"🔥 Starting Fit (Initial Epoch: {start_epoch})")
print(f"📊 Loss trends will be saved to: {log_path}")

history = model.fit(
    train_gen,
    epochs=30,
    initial_epoch=start_epoch,
    callbacks=callbacks
)

print("\n🎉 SESSION FINISHED.")